# US 8 — Modeling-ready data frame

**Sprint 5** | See `docs/modeling_spec.md`

Run **Run All** from repo root or `notebooks/modeling/`.

| Task | Section |
|------|--------|
| 1 | Load balanced dataset |
| 2 | Select target and features |
| 3 | Keep `country` and `year` |
| 4 | Log row count and summary |
| 5 | Check non-null and duplicates |
| 6 | Save `modeling_ready.csv` for US 9 |

**Output:** `data/processed/modeling_ready.csv` — input for US 9 (train/test split per `docs/modeling_spec.md`).

In [1]:
from pathlib import Path

import pandas as pd


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "processed").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "data" / "processed").is_dir():
        return cwd.parent
    if cwd.name == "modeling" and (cwd.parent.parent / "data" / "processed").is_dir():
        return cwd.parent.parent
    return cwd


REPO_ROOT = _repo_root()
DATA_PATH = REPO_ROOT / "data" / "processed" / "final_merged_balanced_adjusted.csv"
OUT_PATH = REPO_ROOT / "data" / "processed" / "modeling_ready.csv"

TARGET = "electricity_demand_per_capita"
FEATURES = [
    "temperature_change_c",
    "co2_per_capita",
    "gdp",
    "population",
    "renewables_share_elec",
    "fossil_share_elec",
]
ID_COLS = ["country", "year"]
MODEL_COLS = [TARGET] + FEATURES
KEEP_COLS = ID_COLS + MODEL_COLS

print("Repo root:", REPO_ROOT)
print("Input:", DATA_PATH)
print("Output:", OUT_PATH)

Repo root: C:\Users\ritup\OneDrive\Desktop\climate energy project\Impact-of-Climate-Change-on-Energy-Demand
Input: C:\Users\ritup\OneDrive\Desktop\climate energy project\Impact-of-Climate-Change-on-Energy-Demand\data\processed\final_merged_balanced_adjusted.csv
Output: C:\Users\ritup\OneDrive\Desktop\climate energy project\Impact-of-Climate-Change-on-Energy-Demand\data\processed\modeling_ready.csv


## Task 1 — Load balanced dataset from `data/processed/`

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"File not found:\n  {DATA_PATH}")

raw = pd.read_csv(DATA_PATH)
print("Task 1 DONE — loaded")
print("Shape:", raw.shape)
raw.head(3)

Task 1 DONE — loaded
Shape: (3300, 22)


,country,year,population,gdp,electricity_demand_per_capita,electricity_demand,electricity_generation,energy_per_capita,energy_per_gdp,fossil_share_elec,...,co2,co2_per_capita,co2_per_gdp,co2_per_unit_energy,consumption_co2,consumption_co2_per_capita,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_c,iso_code
0,Afghanistan,2001,20284303.0,1.102127e+10,38.453377,0.78,0.69,179.891907,0.331085,27.536232,...,1.069098,0.052706,0.097003,0.292985,NaN,NaN,0.000506,0.000904,1.292,AFG
1,Afghanistan,2002,21378123.0,1.880487e+10,37.889202,0.81,0.71,155.695435,0.177001,21.126762,...,1.341065,0.062731,0.071315,0.402907,NaN,NaN,0.000509,0.000911,1.399,AFG
2,Afghanistan,2003,22733053.0,2.107434e+10,44.428699,1.01,0.91,174.557922,0.188297,30.769230,...,1.559679,0.068608,0.074008,0.393041,NaN,NaN,0.000512,0.000920,0.593,AFG


## Task 2 — Select target and feature columns

In [3]:
missing = [c for c in KEEP_COLS if c not in raw.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

print("Task 2 DONE — columns defined")
print("Target:", TARGET)
print("Features:", FEATURES)

Task 2 DONE — columns defined
Target: electricity_demand_per_capita
Features: ['temperature_change_c', 'co2_per_capita', 'gdp', 'population', 'renewables_share_elec', 'fossil_share_elec']


## Task 3 — Keep panel keys (`country`, `year`)

In [4]:
model_df = raw[KEEP_COLS].copy()

print("Task 3 DONE — modeling frame created")
print("Shape:", model_df.shape)
print("Columns:", list(model_df.columns))
model_df.head(3)

Task 3 DONE — modeling frame created
Shape: (3300, 9)
Columns: ['country', 'year', 'electricity_demand_per_capita', 'temperature_change_c', 'co2_per_capita', 'gdp', 'population', 'renewables_share_elec', 'fossil_share_elec']


,country,year,electricity_demand_per_capita,temperature_change_c,co2_per_capita,gdp,population,renewables_share_elec,fossil_share_elec
0,Afghanistan,2001,38.453377,1.292,0.052706,1.102127e+10,20284303.0,72.463768,27.536232
1,Afghanistan,2002,37.889202,1.399,0.062731,1.880487e+10,21378123.0,78.873245,21.126762
2,Afghanistan,2003,44.428699,0.593,0.068608,2.107434e+10,22733053.0,69.230766,30.769230


## Task 4 — Log row count and summary statistics

In [5]:
print("Task 4 — row counts")
print("Rows:", len(model_df))
print("Countries:", model_df["country"].nunique())
print("Years:", int(model_df["year"].min()), "–", int(model_df["year"].max()))
print("\nTask 4 DONE — summary (model columns)")
model_df[MODEL_COLS].describe().T

Task 4 — row counts
Rows: 3300
Countries: 150
Years: 2001 – 2022

Task 4 DONE — summary (model columns)


,count,mean,std,min,25%,50%,75%,max
electricity_demand_per_capita,3300.0,3.933516e+03,5.766188e+03,8.889736e+00,5.058183e+02,2.094885e+03,5.116784e+03,5.604873e+04
temperature_change_c,3300.0,1.112429e+00,5.408282e-01,-4.800000e-01,7.340000e-01,1.041000e+00,1.418000e+00,3.697000e+00
co2_per_capita,3300.0,5.006779e+00,6.690059e+00,2.209881e-02,7.003053e-01,2.797312e+00,6.657652e+00,6.772694e+01
gdp,3300.0,6.052106e+11,2.030520e+12,3.233673e+08,2.603799e+10,8.260384e+10,3.636967e+11,2.696602e+13
population,3300.0,4.578620e+07,1.553772e+08,6.684700e+04,4.082734e+06,1.032527e+07,3.136121e+07,1.426437e+09
renewables_share_elec,3300.0,3.498949e+01,3.340670e+01,0.000000e+00,5.733521e+00,2.313320e+01,6.094755e+01,1.000000e+02
fossil_share_elec,3300.0,6.008371e+01,3.416956e+01,0.000000e+00,3.174059e+01,6.625473e+01,9.245702e+01,1.000000e+02


## Task 5 — Check non-null values and duplicate keys

In [6]:
n_dup = int(model_df.duplicated(subset=ID_COLS).sum())
print("Duplicate country-year rows:", n_dup)

nulls = model_df.isna().sum()
print("\nNull counts per column:")
print(nulls.to_string())

assert n_dup == 0, "Fix duplicate keys before saving"
assert nulls.sum() == 0, "Fix missing values before saving"

print("\nTask 5 DONE — data is clean")

Duplicate country-year rows: 0

Null counts per column:
country                          0
year                             0
electricity_demand_per_capita    0
temperature_change_c             0
co2_per_capita                   0
gdp                              0
population                       0
renewables_share_elec            0
fossil_share_elec                0

Task 5 DONE — data is clean


## Task 6 — Save modeling frame for US 9

In [7]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(OUT_PATH, index=False)

print("Task 6 DONE — saved modeling-ready dataset")
print("Output:", OUT_PATH)
print("Shape:", model_df.shape)
print("Next step (US 9): time-based train/test split per docs/modeling_spec.md")

Task 6 DONE — saved for US 9
Path: C:\Users\ritup\OneDrive\Desktop\climate energy project\Impact-of-Climate-Change-on-Energy-Demand\data\processed\modeling_ready.csv
Shape: (3300, 9)

US 8 complete — Yuvraj can use this file for train/test split (US 9).
